# Vault RAG — Knowledge Graph Retrieval

## Problem with naive RAG

The naive approach dumps the **entire product CSV** into the LLM context for every question:

```
262 products × full descriptions → ~70,000 tokens per question
```

## This approach: vault as a knowledge graph + course pattern

Based on lessons 3.3 and 4.1, we follow the two-step retrieval pattern:

```
Step 1: LLM reads ONLY a short index (surface/category names)
        → returns relevant page names as a pydantic model
Step 2: Load only those vault pages
        → LLM recommends from focused context (~2–5k tokens)
```

**Key tools (from course):**
- `instructor.patch(client)` — structured output via pydantic, no manual `json.loads`
- `response_model=MyModel` — LLM returns a validated pydantic object directly
- `MAP_KNOWLEDGE` — text block embedded in system prompt describing vault rules/structure

**Token savings:**
```
~70,000 tokens (naive)  →  ~2,000–5,000 tokens (vault RAG)
```

In [38]:
%pip install instructor

Note: you may need to restart the kernel to use updated packages.


In [39]:
import os
import re
import time
import json
from pathlib import Path
from datetime import datetime
from typing import List

from openai import OpenAI
from pydantic import BaseModel, Field, conlist
import instructor

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")
MODEL    = "openai/gpt-5.4-mini"  
#MODEL    = "google/gemma-4-31b-it"
#MODEL = "google/gemini-3.1-flash-lite"
BASE_URL = "https://openrouter.ai/api/v1"

BASE_DIR       = Path(".")
VAULT_DIR      = BASE_DIR / "zurada_vault"
QUESTIONS_FILE = BASE_DIR / "../5_navie_rag_extended_data/extended_golden_questions.json"

# instructor.from_openai → modern API, adds create_with_completion for usage tracking
# OpenRouter requires JSON or TOOLS mode (MD_JSON not supported via from_openai)
_raw_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=BASE_URL)
client = instructor.from_openai(_raw_client, mode=instructor.Mode.JSON)

with open(QUESTIONS_FILE, encoding="utf-8") as f:
    questions = json.load(f)

print(f"Model: {MODEL}")
print(f"Vault: {VAULT_DIR}")
print(f"Questions: {len(questions)}")
print(f"API key: {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK'}")

Model: openai/gpt-5.4-mini
Vault: zurada_vault
Questions: 9
API key: OK


---
## MAP_KNOWLEDGE + Pydantic models

**Course pattern (lesson 3.4):** the `Field(description=...)` IS the prompt for each field.
Instead of injecting 300+ names as a flat list into the prompt body, we put **semantic descriptions with examples** directly in the field definition. Instructor passes these to the model as part of the JSON schema.

Three techniques applied:
- `MethodEnum` — closed set of 22 valid method names; pydantic rejects anything outside it and instructor retries
- Rich `Field(description=...)` — top-12 surfaces/categories with semantic hints, so "toaleta" maps to "toalety"
- Explicit defaults — `"Domyślnie zwróć pustą listę"` prevents the model from guessing when uncertain

In [40]:
MAP_KNOWLEDGE = """
Vault Zurada to baza wiedzy produktów czyszczących, zorganizowana jako mapa połączonych stron.

Struktura vaultu:
- Powierzchnie/  — strony dla każdej powierzchni (np. toalety, podłogi, szkło)
- Kategorie/     — strony dla każdej kategorii (np. wc, zele-do-wc, odkamienianie)
- Metody/        — metody mycia (np. ręczne, myjnia, pianowanie)
- Produkty/      — pełne opisy produktów (frontmatter: product_id, ph, metody, kategorie)

Zasady odpowiedzi:
- Odpowiadaj WYŁĄCZNIE na podstawie dostarczonych stron produktów
- Nie wymyślaj cech produktów spoza dostarczonych danych
- Odróżniaj formy (żel vs piana), przeznaczenie (domowe vs przemysłowe), metodę (ręczne vs maszynowe)
"""


class ProductCandidate(BaseModel):
    product_id: int
    reason: str = Field(description="Jedno zdanie uzasadnienia")


class RecommendModel(BaseModel):
    chain_of_thought: List[str]         = Field(default_factory=list)
    product_ids: List[ProductCandidate] = Field(default_factory=list)


print("MAP_KNOWLEDGE and RecommendModel defined")
print("IntentModel will be defined after vault index is loaded")

MAP_KNOWLEDGE and RecommendModel defined
IntentModel will be defined after vault index is loaded


---
## Step 0 — Build vault index

Parse the vault once at startup: build maps of `surface/category → product names`.
No LLM needed here — pure file parsing.

In [41]:
def parse_product_links(md_text: str) -> list[str]:
    return re.findall(r'\[\[Produkty/([^|\]]+)[|\]]', md_text)


def parse_frontmatter_field(md_text: str, field: str):
    m = re.search(rf'^{field}:\s*(.+)$', md_text, re.MULTILINE)
    return m.group(1).strip() if m else None


name_to_id: dict[str, int] = {}
for f in (VAULT_DIR / "Produkty").glob("*.md"):
    text = f.read_text(encoding="utf-8")
    pid  = parse_frontmatter_field(text, "product_id")
    if pid:
        name_to_id[f.stem] = int(pid)

surfaces_index: dict[str, list[str]] = {}
for f in (VAULT_DIR / "Powierzchnie").glob("*.md"):
    surfaces_index[f.stem] = parse_product_links(f.read_text(encoding="utf-8"))

categories_index: dict[str, list[str]] = {}
for f in (VAULT_DIR / "Kategorie").glob("*.md"):
    categories_index[f.stem] = parse_product_links(f.read_text(encoding="utf-8"))

methods_index: dict[str, list[str]] = {}
for f in (VAULT_DIR / "Metody").glob("*.md"):
    methods_index[f.stem] = parse_product_links(f.read_text(encoding="utf-8"))

print(f"Products: {len(name_to_id)}  Surfaces: {len(surfaces_index)}  "
      f"Categories: {len(categories_index)}  Methods: {len(methods_index)}")

Products: 262  Surfaces: 307  Categories: 285  Methods: 22


In [42]:
import enum as _enum

# --- helpers ---

def _page_description(path: Path) -> str:
    text  = path.read_text(encoding="utf-8")
    parts = text.split("---", 2)
    body  = parts[2] if len(parts) >= 3 else text
    for line in body.strip().split("\n"):
        line = line.strip()
        if line and not line.startswith("#") and not line.startswith("-"):
            return line[:100]
    return ""


def _make_enum(name: str, keys: list[str]) -> type:
    """Build a str-enum from vault page names.
    Pydantic validates returned values against the enum;
    instructor retries automatically when the model returns an invalid name.
    """
    return _enum.Enum(
        name,
        {k.upper().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", ""): k
         for k in sorted(keys)},
        type=str,
    )


# --- Three closed-set enums (lesson 3.4 pattern) ---
# Pro: exact-match validation + automatic retry on invalid values
# Cost: instructor embeds the full enum schema in the prompt
#   MethodEnum   22 values  →  negligible token cost
#   SurfaceEnum  307 values →  ~1,500 extra tokens per call
#   CategoryEnum 285 values →  ~1,400 extra tokens per call

MethodEnum   = _make_enum("MethodEnum",   list(methods_index.keys()))
SurfaceEnum  = _make_enum("SurfaceEnum",  list(surfaces_index.keys()))
CategoryEnum = _make_enum("CategoryEnum", list(categories_index.keys()))

_method_desc = "; ".join(
    f"{m.value}: {_page_description(VAULT_DIR / 'Metody' / f'{m.value}.md')}"
    for m in MethodEnum
)


# --- IntentModel with all three enums ---
class IntentModel(BaseModel):
    surfaces: List[SurfaceEnum] = Field(
        default_factory=list,
        description=(
            "Powierzchnie fizyczne wymienione lub sugerowane w pytaniu. "
            "Domyślnie zwróć pustą listę jeśli brak jednoznacznego dopasowania."
        ),
    )
    categories: List[CategoryEnum] = Field(
        default_factory=list,
        description=(
            "Kategorie produktów pasujące do pytania. "
            "Domyślnie zwróć pustą listę jeśli brak jednoznacznego dopasowania."
        ),
    )
    methods: List[MethodEnum] = Field(
        default_factory=list,
        description=(
            "Metody aplikacji wymagane w pytaniu. "
            "Wybierz tylko jeśli pytanie jawnie wskazuje metodę. "
            "Domyślnie zwróć pustą listę. "
            f"Dostępne: {_method_desc}"
        ),
    )
    reasoning_steps: List[str] = Field(default_factory=list)


print(f"SurfaceEnum:  {len(list(SurfaceEnum))} values")
print(f"CategoryEnum: {len(list(CategoryEnum))} values")
print(f"MethodEnum:   {len(list(MethodEnum))} values")
print("All three use closed-set validation → instructor retries on invalid names")

SurfaceEnum:  307 values
CategoryEnum: 285 values
MethodEnum:   22 values
All three use closed-set validation → instructor retries on invalid names


---
## Step 1 — Extract intent from the question

A small, cheap LLM call that asks: *what surfaces, categories and methods does this question relate to?*  
The LLM returns keys that map directly to vault page names.

In [43]:
INTENT_SYSTEM_3 = """Jesteś analitykiem zapytań dla bazy wiedzy produktów czyszczących Zurada.

{MAP_KNOWLEDGE}

Na podstawie pytania klienta zidentyfikuj, które strony vaultu są potrzebne.
ZASADY KRYTYCZNE:
1. UŻYWAJ WYŁĄCZNIE WARTOŚCI Z LISTY. Nie wymyślaj własnych słów (np. nie używaj słowa "muszla", jeśli nie ma go w dostępnym Enumie).
2. ZACHOWAJ ORYGINALNĄ PISOWNIĘ Z ENUMÓW. Jeśli kategoria w schemacie nie ma polskich znaków (np. "reczne-mycie-naczyn"), zwróć ją DOKŁADNIE w takiej formie, bez "ę", "ń" itp.
3. KATEGORYZUJ POPRAWNIE: 
   - 'surfaces' to fizyczne obiekty (np. 'toalety', 'blaty'). Nigdy nie wkładaj tu procesów ani typów chemii.
   - 'categories' to grupy chemii domowej/warsztatowej.
Wybierz od 1 do maksymalnie 6 najbardziej pasujących pozycji per pole."""

def extract_intent(question: str) -> tuple[IntentModel, object]:
    intent, completion = client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_3.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)},
            {"role": "user",   "content": question},
        ],
        response_model=IntentModel,
        max_tokens=4000,
        temperature=0,
    )
    return intent, completion.usage


demo_q = questions[0]
intent, usage_demo = extract_intent(demo_q["question"])
print(f"Question:   {demo_q['question']}")
print(f"Surfaces:   {intent.surfaces}")
print(f"Categories: {intent.categories}")
print(f"Methods:    {[m.value for m in intent.methods]}")
print(f"Reasoning:  {intent.reasoning_steps}")
print(f"Usage:      prompt={usage_demo.prompt_tokens}  completion={usage_demo.completion_tokens}  total={usage_demo.total_tokens}")

Question:   Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
Surfaces:   [<SurfaceEnum.TOALETY: 'toalety'>, <SurfaceEnum.CERAMIKA_SANITARNA: 'ceramika sanitarna'>]
Categories: [<CategoryEnum.ZELE_DO_WC: 'zele-do-wc'>, <CategoryEnum.HIGIENA_TOALET: 'higiena-toalet'>, <CategoryEnum.ODKAMIENIANIE: 'odkamienianie'>, <CategoryEnum.SRODKI_DO_LAZIENKI: 'srodki-do-lazienki'>, <CategoryEnum.CHEMIA_DOMOWA: 'chemia-domowa'>, <CategoryEnum.DLA_DOMU: 'dla-domu'>]
Methods:    []
Reasoning:  ['Pytanie dotyczy toalety w domu, więc pasują powierzchnie związane z toaletą i ceramiką sanitarną.', "Sformułowanie 'zwykły, gęsty żel' wskazuje na produkty z kategorii żeli do WC.", 'Wzmianka o usuwaniu kamienia pasuje do odkamieniania.', "'Odświeżyć' wskazuje na higienę toalet.", "'W domu' uzasadnia kategorie dla chemii domowej oraz dla domu.", 'Nie ma jawnie wskazanej metody aplikacji, więc methods pozostaje puste.']
Usage:      prompt=7302

---
## Step 2 — Look up vault pages

Use the intent to find candidate products.  
No LLM — pure index lookup using the vault's linked structure.

In [44]:
def lookup_candidates(intent: IntentModel) -> set[str]:
    """Step 2: pure Python vault lookup — all enum values use .value to get the string key."""
    candidates: set[str] = set()

    for surface in intent.surfaces:
        candidates.update(surfaces_index.get(surface.value, []))

    for cat in intent.categories:
        candidates.update(categories_index.get(cat.value, []))

    for method in intent.methods:
        if not candidates:  # methods are broad — only use as fallback
            candidates.update(methods_index.get(method.value, []))

    return candidates


candidates = lookup_candidates(intent)
print(f"Candidate products ({len(candidates)}): ")
candidates_id = [name_to_id[c] for c in candidates if c in name_to_id]
for c in sorted(candidates_id):
    print(f"- {c}")

Candidate products (63): 
- 4
- 10
- 11
- 12
- 25
- 27
- 29
- 33
- 41
- 46
- 51
- 68
- 79
- 80
- 82
- 92
- 93
- 94
- 127
- 128
- 129
- 130
- 131
- 133
- 134
- 135
- 136
- 137
- 138
- 139
- 156
- 165
- 188
- 189
- 190
- 211
- 212
- 213
- 214
- 215
- 216
- 217
- 218
- 219
- 220
- 221
- 222
- 223
- 224
- 225
- 226
- 227
- 228
- 229
- 230
- 231
- 234
- 235
- 236
- 237
- 238
- 239
- 240


---
## Step 3 — Read candidate product pages

Load only the `.md` files for the candidate products from the vault.  
Each page contains the full product description, pH, surfaces, safety notes — everything the LLM needs.

In [45]:
def load_product_pages(product_names: set[str]) -> str:
    """Read vault pages for the given product names and return them as one text block."""
    pages = []
    for name in sorted(product_names):
        path = VAULT_DIR / "Produkty" / f"{name}.md"
        if path.exists():
            pages.append(path.read_text(encoding="utf-8"))
    return "\n\n---\n\n".join(pages)


product_context = load_product_pages(candidates)
chars  = len(product_context)
approx = chars // 4
print(f"Loaded {len(candidates)} product pages")
print(f"Context size: {chars:,} chars (~{approx:,} tokens)")
print(f"Naive CSV was: ~70,000 tokens → saving ~{70000 - approx:,} tokens ({(70000-approx)/70000*100:.0f}%)")

Loaded 63 product pages
Context size: 116,510 chars (~29,127 tokens)
Naive CSV was: ~70,000 tokens → saving ~40,873 tokens (58%)


---
## Step 4 — Ask LLM with only the relevant products

The LLM now sees a focused, small context instead of the entire catalog.

In [46]:
RECOMMEND_SYSTEM = """Jesteś ekspertem ds. środków czystości firmy Zurada.

{MAP_KNOWLEDGE}

Poniżej otrzymasz wybrane strony produktów z bazy wiedzy Zurada w formacie Markdown.
Na podstawie tych stron odpowiedz na pytanie klienta — wskaż pasujące produkty z ich product_id."""

RECOMMEND_SYSTEM_2 = """Jesteś rygorystycznym ekspertem ds. środków czystości firmy Zurada.

{MAP_KNOWLEDGE}

Poniżej otrzymasz wybrane strony produktów z bazy wiedzy Zurada w formacie Markdown.
Twoim zadaniem jest trafna odpowiedź na pytanie klienta, wybierając odpowiednie produkty TYLKO z dostarczonego kontekstu.

Zasady analizy (Logika i Bezpieczeństwo):
1. Opieraj się w 100% na faktach z dostarczonych stron. Nie wymyślaj przeznaczenia produktu.
2. Zwracaj uwagę na wykluczenia ('odradzane_powierzchnie'). Używaj logiki chemicznej (np. marmur to wrażliwy kamień naturalny, unikaj kwasów). Jeśli dany środek jest wyraźnie niebezpieczny dla klienta - pomiń go.
3. Czytaj intencję klienta: odróżniaj środki do mycia ręcznego od maszynowego (zmywarki) oraz formy aplikacji (żel vs piana).
4. Dopasuj kontekst i skalę zastosowania: Odróżniaj codzienne zastosowania domowe od specjalistycznej chemii przemysłowej, warsztatowej lub gastronomicznej (HoReCa).
5. Jeśli dany środek ogólnie pasuje do czyszczonej powierzchni i nie ma wyraźnych przeciwwskazań, możesz go polecić.

Jeśli żaden produkt nie pasuje lub użycie każdego grozi zniszczeniem powierzchni — zwróć pustą listę product_ids."""

def recommend(question: str, product_context: str) -> tuple[RecommendModel, object]:
    """Step 4: focused LLM call → returns (RecommendModel, usage)."""
    system = RECOMMEND_SYSTEM_2.format(MAP_KNOWLEDGE=MAP_KNOWLEDGE)
    user_msg = f"Strony produktów:\n\n{product_context}\n\n---\n\nPytanie klienta: {question}"

    result, completion = client.chat.completions.create_with_completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user_msg},
        ],
        response_model=RecommendModel,
        max_tokens=4000,
        temperature=0,
    )
    return result, completion.usage


result, usage_rec = recommend(demo_q["question"], load_product_pages(candidates))

returned_ids = [c.product_id for c in result.product_ids]
print(f"Question:  {demo_q['question']}")
print(f"Expected:  {demo_q['expectedProductIds']}")
print(f"Returned:  {returned_ids}")
print(f"Usage:     prompt={usage_rec.prompt_tokens}  completion={usage_rec.completion_tokens}  total={usage_rec.total_tokens}")
for c in result.product_ids:
    print(f"  [{c.product_id}] {c.reason}")

Question:  Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
Expected:  [12, 82, 139]
Returned:  [139, 12, 82, 79, 80]
Usage:     prompt=42096  completion=390  total=42486
  [139] To gęsty, kwaśny żel do ręcznego mycia toalet, usuwa kamień i rdzę oraz jest przeznaczony do codziennego czyszczenia muszli w domu.
  [12] To zagęszczony żel do ręcznego czyszczenia toalet i sanitariatów, skutecznie usuwa kamień i osady, więc dobrze pasuje do domowego WC.
  [82] To ekologiczny, skoncentrowany żel do mycia toalet i odkamieniania sanitariatów, odpowiedni do bieżącego utrzymania czystości muszli.
  [79] To chlorowy żel do ręcznego mycia sanitariatów, który czyści i wybiela toalety, ale należy unikać aluminium i powierzchni wrażliwych na chlor.
  [80] To zagęszczony żel do mycia toalet usuwający silny kamień i rdzę, więc sprawdzi się, gdy w muszli jest więcej osadu.


---
## Full pipeline — run all questions

In [47]:
def vault_rag_pipeline(question: str) -> tuple[list[int], dict]:
    """Full pipeline: intent → vault lookup → product pages → recommend. Returns (ids, stats)."""
    t0 = time.perf_counter()

    intent, usage_intent = extract_intent(question)

    candidates      = lookup_candidates(intent)
    product_context = load_product_pages(candidates)

    rec, usage_rec = recommend(question, product_context)

    elapsed = time.perf_counter() - t0

    returned_ids = [c.product_id for c in rec.product_ids]
    stats = {
        "intent":             {"surfaces": [s.value for s in intent.surfaces],
                               "categories": [c.value for c in intent.categories],
                               "methods": [m.value for m in intent.methods]},
        "n_candidates":       len(candidates),
        "n_tokens_est":       len(product_context) // 4,
        "prompt_tokens":      usage_intent.prompt_tokens     + usage_rec.prompt_tokens,
        "completion_tokens":  usage_intent.completion_tokens + usage_rec.completion_tokens,
        "total_tokens":       usage_intent.total_tokens      + usage_rec.total_tokens,
        "time_seconds":       round(elapsed, 2),
    }
    return returned_ids, stats


results = []

for q in questions:
    qid      = q["id"]
    expected = set(q["expectedProductIds"])

    try:
        returned_ids, stats = vault_rag_pipeline(q["question"])
        returned = set(returned_ids)
        error = None
    except Exception as e:
        returned, stats, error = set(), {"n_candidates": 0, "intent": {}, "time_seconds": 0,
                                         "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}, str(e)

    hit     = expected & returned
    missed  = expected - returned
    extra   = returned - expected
    correct = returned == expected

    results.append({
        "id":                qid,
        "difficulty":        q.get("difficulty", ""),
        "expected":          sorted(expected),
        "returned":          sorted(returned),
        "hit":               sorted(hit),
        "missed":            sorted(missed),
        "extra":             sorted(extra),
        "correct":           correct,
        "n_candidates":      stats.get("n_candidates", 0),
        "n_tokens_est":      stats.get("n_tokens_est", 0),
        "prompt_tokens":     stats.get("prompt_tokens", 0),
        "completion_tokens": stats.get("completion_tokens", 0),
        "total_tokens":      stats.get("total_tokens", 0),
        "time_seconds":      stats.get("time_seconds", 0),
        "intent":            stats.get("intent", {}),
        "error":             error,
    })

    status = "✓" if correct else "✗"
    print(f"[{status}] Q{qid:02d} [{q['difficulty']:4s}] "
          f"candidates={stats.get('n_candidates',0):3d}  "
          f"tokens={stats.get('total_tokens',0):5,}  "
          f"time={stats.get('time_seconds',0):.1f}s  "
          f"expected={sorted(expected)}  returned={sorted(returned)}")

correct_total = sum(r["correct"] for r in results)
print(f"\n{correct_total}/{len(results)} exact matches  "
      f"avg time={sum(r['time_seconds'] for r in results)/len(results):.1f}s  "
      f"avg tokens={sum(r['total_tokens'] for r in results)/len(results):,.0f}")

[✗] Q01 [easy] candidates= 61  tokens=48,650  time=6.4s  expected=[12, 82, 139]  returned=[12, 79, 82, 139]
[✗] Q08 [easy] candidates= 71  tokens=52,946  time=8.8s  expected=[136, 137, 189, 190]  returned=[72, 136, 137, 189, 190]
[✓] Q09 [easy] candidates= 15  tokens=18,473  time=4.0s  expected=[233, 257]  returned=[233, 257]
[✗] Q10 [easy] candidates= 21  tokens=21,923  time=4.0s  expected=[213, 217, 218]  returned=[213, 218]
[✗] Q02 [easy] candidates=  7  tokens=14,550  time=4.9s  expected=[5, 34, 128]  returned=[5, 34, 96, 128]
[✓] Q03 [hard] candidates= 86  tokens=70,105  time=8.9s  expected=[45, 112, 152]  returned=[45, 112, 152]
[✗] Q04 [hard] candidates= 31  tokens=34,375  time=6.3s  expected=[11]  returned=[]
[✗] Q05 [easy] candidates= 58  tokens=44,107  time=8.7s  expected=[2, 39, 41, 129]  returned=[41, 129]
[✓] Q06 [easy] candidates= 60  tokens=47,812  time=6.1s  expected=[7, 69]  returned=[7, 69]

3/9 exact matches  avg time=6.4s  avg tokens=39,216


---
## Results summary

In [48]:
import pandas as pd

df = pd.DataFrame(results)

n_correct = df["correct"].sum()
n_total   = len(df)

print("=" * 65)
print(f"RESULT: {n_correct}/{n_total} exact matches ({n_correct/n_total*100:.1f}%)")
print(f"Avg candidates/q:   {df['n_candidates'].mean():>8.1f}  (out of 262 total)")
print(f"Avg ~tokens/q:      {df['n_tokens_est'].mean():>8,.0f}  (naive: ~70,000)")
print("=" * 65)
print()

display_cols = ["id", "difficulty", "expected", "returned", "correct", "missed", "extra", "n_candidates"]
print(df[display_cols].to_string(index=False))
print()

errors = df[~df["correct"]]
if not errors.empty:
    print(f"Wrong answers ({len(errors)}):\n")
    for _, row in errors.iterrows():
        print(f"  Q{row['id']:02d} [{row['difficulty']}]")
        print(f"    expected:   {row['expected']}")
        print(f"    returned:   {row['returned']}")
        print(f"    missed:     {row['missed']}")
        print(f"    extra:      {row['extra']}")
        print(f"    candidates: {row['n_candidates']}  intent: {row['intent']}")

RESULT: 3/9 exact matches (33.3%)
Avg candidates/q:       45.6  (out of 262 total)
Avg ~tokens/q:        21,666  (naive: ~70,000)

 id difficulty             expected                 returned  correct  missed extra  n_candidates
  1       easy        [12, 82, 139]        [12, 79, 82, 139]    False      []  [79]            61
  8       easy [136, 137, 189, 190] [72, 136, 137, 189, 190]    False      []  [72]            71
  9       easy           [233, 257]               [233, 257]     True      []    []            15
 10       easy      [213, 217, 218]               [213, 218]    False   [217]    []            21
  2       easy         [5, 34, 128]         [5, 34, 96, 128]    False      []  [96]             7
  3       hard       [45, 112, 152]           [45, 112, 152]     True      []    []            86
  4       hard                 [11]                       []    False    [11]    []            31
  5       easy     [2, 39, 41, 129]                [41, 129]    False [2, 39]    []  

---
## Save results for metrics analysis

In [49]:
N_RUNS      = 8
OUTPUT_FILE = BASE_DIR / "vault_rag_runs.json"

repeated = []

for q in questions:
    qid  = q["id"]
    runs = []

    for run_idx in range(1, N_RUNS + 1):
        try:
            returned_ids, stats = vault_rag_pipeline(q["question"])
            runs.append({
                "run":               run_idx,
                "returned_ids":      returned_ids,
                "model":             MODEL,
                "n_candidates":      stats["n_candidates"],
                "prompt_tokens":     stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "total_tokens":      stats["total_tokens"],
                "time_seconds":      stats["time_seconds"],
                "intent":            stats["intent"],
            })
        except Exception as e:
            runs.append({"run": run_idx, "error": str(e)})

    times  = [r["time_seconds"]  for r in runs if "time_seconds"  in r]
    tokens = [r["total_tokens"]  for r in runs if "total_tokens"   in r]

    avg_time   = round(sum(times)  / len(times),  2) if times  else None
    avg_tokens = round(sum(tokens) / len(tokens), 0) if tokens else None

    repeated.append({
        "id":                   qid,
        "question":             q["question"],
        "expectedProductIds":   q["expectedProductIds"],
        "expectedNoProductIds": q["expectedNoProductIds"],
        "difficulty":           q.get("difficulty", ""),
        "avg_time_seconds":     avg_time,
        "avg_total_tokens":     avg_tokens,
        "runs":                 runs,
    })

    expected_set = set(q["expectedProductIds"])
    n_correct    = sum(1 for r in runs if set(r.get("returned_ids", [])) == expected_set)
    print(f"Q{qid:02d} [{q['difficulty']:4s}] {n_correct}/{N_RUNS} correct  "
          f"avg={avg_time:.1f}s  avg_tokens={avg_tokens:,.0f}")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "model":     MODEL,
        "approach":  "vault_rag",
        "n_runs":    N_RUNS,
        "questions": repeated,
    }, f, ensure_ascii=False, indent=2)

print(f"\nSaved: {OUTPUT_FILE}")

Q01 [easy] 2/8 correct  avg=4.7s  avg_tokens=48,722
Q08 [easy] 5/8 correct  avg=6.1s  avg_tokens=50,809
Q09 [easy] 8/8 correct  avg=5.5s  avg_tokens=22,215
Q10 [easy] 0/8 correct  avg=4.3s  avg_tokens=19,368
Q02 [easy] 5/8 correct  avg=8.3s  avg_tokens=44,909
Q03 [hard] 2/8 correct  avg=6.7s  avg_tokens=70,169
Q04 [hard] 0/8 correct  avg=5.6s  avg_tokens=38,272
Q05 [easy] 0/8 correct  avg=4.3s  avg_tokens=46,858
Q06 [easy] 7/8 correct  avg=4.0s  avg_tokens=45,149

Saved: vault_rag_runs.json
